# Quick Retrain — Dev/Debug

Train nhanh 3 models trên **sample 10K dòng** để kiểm tra pipeline.
Không ghi đè production models (lưu vào `models/dev_*`).

**Input**: `data/it_jobs_processed.csv`
**Output**: `models/dev_salary_model.joblib`, `models/dev_demand_model.joblib`, `models/dev_cluster_model.joblib`

In [ ]:
import os, joblib
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_absolute_error

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_FILE = os.path.join(BASE_DIR, 'data', 'it_jobs_processed.csv')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
SAMPLE_SIZE = 10_000  # Giảm xuống 1000 nếu muốn chạy siêu nhanh

SAVE_PREFIX = 'dev_'  # Không ghi đè production models

In [ ]:
df = pd.read_csv(DATA_FILE).sample(min(SAMPLE_SIZE, 100000), random_state=42)
print(f'Loaded sample: {len(df)} rows')

features = ['num_skills', 'skill_diversity', 'skill_programming', 'skill_cloud', 'skill_ai_ml',
            'skill_database', 'skill_devops', 'skill_framework', 'skill_data_engineering',
            'skill_security', 'skill_soft_skills', 'seniority_level', 'job_type', 'state', 'it_domain']
numeric_features = [f for f in features if f.startswith('skill_') or f == 'num_skills']
categorical_features = ['seniority_level', 'job_type', 'state', 'it_domain']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])

In [ ]:
# 1. Salary
salary_df = df.dropna(subset=['salary_annual'])
if len(salary_df) >= 50:
    X = salary_df[features]; y = salary_df['salary_annual']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    pipe = Pipeline([('pre', preprocessor), ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])
    pipe.fit(X_tr, y_tr)
    r2, mae = r2_score(y_te, pipe.predict(X_te)), mean_absolute_error(y_te, pipe.predict(X_te))
    print(f'Salary: R²={r2:.4f}, MAE=${mae:,.0f}')
    joblib.dump(pipe, os.path.join(MODELS_DIR, f'{SAVE_PREFIX}salary_model.joblib'))
else:
    print('Salary: insufficient data')

In [ ]:
# 2. Demand
demand_df = df.groupby(['it_domain', 'state', 'seniority_level', 'job_type']).size().reset_index(name='cnt')
demand_df['score'] = np.log1p(demand_df['cnt']) / np.log1p(demand_df['cnt']).max() * 100
if len(demand_df) >= 5:
    X_d = demand_df[['it_domain', 'state', 'seniority_level', 'job_type']]
    y_d = demand_df['score']
    pre_d = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['it_domain', 'state', 'seniority_level', 'job_type'])])
    pipe = Pipeline([('pre', pre_d), ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])
    X_dtr, X_dte, y_dtr, y_dte = train_test_split(X_d, y_d, test_size=0.2, random_state=42)
    pipe.fit(X_dtr, y_dtr)
    print(f'Demand: R²={r2_score(y_dte, pipe.predict(X_dte)):.4f}')
    joblib.dump(pipe, os.path.join(MODELS_DIR, f'{SAVE_PREFIX}demand_model.joblib'))
else:
    print('Demand: insufficient data')

In [ ]:
# 3. Cluster
cluster_df = df.dropna(subset=['salary_annual'])
if len(cluster_df) >= 50:
    X_c = cluster_df[features]
    pipe = Pipeline([('pre', preprocessor), ('pca', PCA(n_components=3, random_state=42)), ('kmeans', KMeans(n_clusters=4, random_state=42, n_init=5))])
    pipe.fit(X_c)
    print(f'Cluster: trained on {len(X_c)} rows')
    joblib.dump(pipe, os.path.join(MODELS_DIR, f'{SAVE_PREFIX}cluster_model.joblib'))
else:
    print('Cluster: insufficient data')

## Kết quả

Các model dev được lưu với prefix `dev_`. Để dùng production models, chạy `training.ipynb` hoặc `retrain_all.py`.

```bash
# Copy dev models lên production nếu cần
copy models\dev_salary_model.joblib models\best_salary_model.joblib
```